# 🧑‍💻 Building the Frontier for AI with Databases - Demos

![AI Databases event](../assets/ai-databases-event.png)


## 1. ⚙️ Preparations


```mermaid
%%{init: {"themeVariables": {"fontSize": "20px"}}}%%
flowchart LR
    A([1.1 Install packages]) --> B([1.2 Import libraries])
    B --> C([1.3 Sign in to Azure])
    C --> D([1.4 Verify identity])
    D --> E([1.5 Load books.csv])
    E --> F([1.6 Inspect the dataset])
```


In [146]:
# Install required packages
# %pip install ipykernel pandas python-dotenv numpy matplotlib seaborn agent-framework azure-mgmt-postgresqlflexibleservers psycopg2-binary

In [147]:
# Import libraries
import os
import json
from dotenv import load_dotenv
import pandas as pd
import textwrap
import re
from IPython.display import Markdown, display

# Load environment variables from .env file
load_dotenv(override=True)
print("✅ Environment variables loaded successfully!")

✅ Environment variables loaded successfully!


In [148]:
# Login using AzureCliCredentials and interactively with browser
from azure.identity import AzureDeveloperCliCredential

credential = AzureDeveloperCliCredential()

print("✅ Azure Developer CLI credentials loaded successfully!", credential.tenant_id)

✅ Azure Developer CLI credentials loaded successfully! 


In [181]:
# Show which identity the credential is actually using (decodes the token claims locally)
import base64

token = credential.get_token("https://management.azure.com/.default").token
payload = token.split(".")[1]
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))

# Resolved here because the PostgreSQL cells reuse it as the login name
signed_in_as = (claims.get("upn") or claims.get("unique_name")
                or claims.get("preferred_username") or claims.get("appid"))

# The value itself is deliberately not printed -- it would bake an identity into the committed notebook
print(f"🔒 Signed in — identity resolved from token claims "
      f"({'user' if signed_in_as and '@' in signed_in_as else 'service principal'}).")


🔒 Signed in — identity resolved from token claims (user).


In [150]:
# Load the books dataset
books_df = pd.read_csv('../data/books.csv')

print("✅ Books dataset loaded successfully!", "Number of records:", len(books_df))

✅ Books dataset loaded successfully! Number of records: 6810


In [151]:
# List column names
print("📊 Columns in the dataset:\n", textwrap.fill(", ".join(books_df.columns.tolist()), width=80))

📊 Columns in the dataset:
 isbn13, isbn10, title, subtitle, authors, categories, thumbnail, description,
published_year, average_rating, num_pages, ratings_count


In [152]:
# Show a few records from the dataset for title, authors, categories, and description
print("📚 Sample records from the dataset:")
books_df[['isbn13', 'title', 'authors', 'categories', 'description']].head()

📚 Sample records from the dataset:


,isbn13,title,authors,categories,description
0,9780002005883,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...
1,9780002261982,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,A new 'Christie for Christmas' -- a full-lengt...
2,9780006163831,The One Tree,Stephen R. Donaldson,American fiction,Volume Two of Stephen Donaldson's acclaimed se...
3,9780006178736,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b..."
4,9780006280897,The Four Loves,Clive Staples Lewis,Christian life,Lewis' work on the nature of love divides love...


In [153]:
# Show the description of a specific book by index
book_index = 0  # Change this index to view a different book
sample_title = books_df.loc[book_index, 'title']
sample_description = books_df.loc[book_index, 'description']
print(f"📘 Title: {sample_title}")
print(f"📝 Description: \n{textwrap.fill(sample_description, width=100)}")

📘 Title: Gilead
📝 Description: 
A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an
astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and
the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end
of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the
young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift
between his grandfather and his father: the elder, an angry visionary who fought for the
abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake,
Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for
forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and
truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the 

---

## 2. 🪐 Azure Cosmos DB


```mermaid
%%{init: {"themeVariables": {"fontSize": "20px"}}}%%
flowchart LR
    A([2.1 Setup]) --> B([2.2 Embeddings])
    B --> C([2.3 Insert data])
    C --> D([2.4 Search features])
    D --> E([2.5 Keyword search])
    D --> F([2.6 Vector search])
    D --> G([2.7 Hybrid search])
    G --> H([2.8 Ground an agent])
```


### 2.1 🛠️ Setup and Configuration


In [154]:
# Prepare the Cosmos DB configurations
from azure.cosmos import CosmosClient, PartitionKey

COSMOS_DB_ENDPOINT = os.getenv("COSMOS_DB_ENDPOINT")
COSMOS_DB_DATABASE = os.getenv("COSMOS_DB_DATABASE")
COSMOS_DB_CONTAINER = os.getenv("COSMOS_DB_CONTAINER")

# Create a CosmosClient instance using the endpoint and credentials
cosmos_client = CosmosClient(COSMOS_DB_ENDPOINT, credential=credential)

# Create a database if it doesn't exist
cosmos_database = cosmos_client.create_database_if_not_exists(id=COSMOS_DB_DATABASE)

print(f"✅ Cosmos database '{COSMOS_DB_DATABASE}' created or already exists.")    

✅ Cosmos database 'books-db' created or already exists.


In [155]:
# Create a container in the database if it doesn't exist

# Define vector embedding policy
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/embeddings",   # JSON path to the vector field
            "dataType": "float32",  # Supported: float32
            "dimensions": 1536,     # Example: OpenAI embedding size
            "distanceFunction": "cosine"  # Supported: cosine, euclidean, dotproduct
        }
    ]
}

# Define full text policy, required by FullTextScore and hybrid search
full_text_policy = {
    "defaultLanguage": "en-US",
    "fullTextPaths": [
        {"path": "/title", "language": "en-US"},
        {"path": "/description", "language": "en-US"}
    ]
}

# Define indexing policy
indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [
        {"path": "/*"}  # Index all properties
    ],
    "vectorIndexes": [
      {
          "path": "/embeddings",
          "type": "quantizedFlat" # Supported: flat, quantizedFlat, diskANN
      }
    ],
    "fullTextIndexes": [
        {"path": "/title"},
        {"path": "/description"}
    ],
    "excludedPaths": [
        {"path": "/_etag/?"}  # Exclude system property
    ]
}

# Create container with policies
cosmos_container = cosmos_database.create_container_if_not_exists(
    id=COSMOS_DB_CONTAINER,
    partition_key=PartitionKey(path="/id"),  # Example partition key
    indexing_policy=indexing_policy,
    vector_embedding_policy=vector_embedding_policy,
    full_text_policy=full_text_policy
)

print(f"✅ Container '{COSMOS_DB_CONTAINER}' created with vector, full text, and indexing policies.")


✅ Container 'books' created with vector, full text, and indexing policies.


#### 2.1.1 🧭 Container Policies
<img src="../assets/cosmosdb-vector-policy.png" alt="Vector Policy" width="300">
<img src="../assets/cosmosdb-full-text-policy.png" alt="Full Text Policy" width="300">


### 2.2 🧮 Generate Embeddings


In [156]:
# Create an embedding client for the Foundry service
from agent_framework.foundry import FoundryEmbeddingClient
from azure.ai.inference.aio import EmbeddingsClient

FOUNDRY_MODELS_ENDPOINT = os.getenv("FOUNDRY_MODELS_ENDPOINT")
FOUNDRY_EMBEDDING_MODEL = os.getenv("FOUNDRY_EMBEDDING_MODEL")

# These clients own an aiohttp session. Closing any earlier one keeps this cell re-runnable --
# otherwise every re-run orphans a session and Python warns "Unclosed client session" later.
if "embedding_client" in globals():
    await embedding_client.close()
if "text_client" in globals():
    await text_client.close()

# The inference SDK defaults to the ml.azure.com audience, which an AI Services endpoint rejects.
text_client = EmbeddingsClient(
    endpoint=FOUNDRY_MODELS_ENDPOINT,
    credential=credential,
    credential_scopes=["https://cognitiveservices.azure.com/.default"],
)

embedding_client = FoundryEmbeddingClient(
    endpoint=FOUNDRY_MODELS_ENDPOINT,
    model=FOUNDRY_EMBEDDING_MODEL,
    credential=credential,
    text_client=text_client,
)

print(f"🤖 Embedding model: {embedding_client.model}")
# Get the embedding for the sample description
sample_embedding_response = await embedding_client.get_embeddings([sample_description])

# Extract the embedding vector from the response
sample_embedding = sample_embedding_response[0]
print(f"🔢 Sample vector dimensions: {sample_embedding.dimensions}")
print(f"🧮 Sample vector:{textwrap.fill(str(sample_embedding.vector[0:10]), width=70)}")


🤖 Embedding model: text-embedding-3-large
🔢 Sample vector dimensions: 3072
🧮 Sample vector:[-0.005535125732421875, 0.0010805130004882812, -0.01480865478515625,
0.0215911865234375, -0.060882568359375, -0.0086822509765625,
-0.015228271484375, -0.0216217041015625, -0.005779266357421875,
0.0278472900390625]


### 2.3 ✍️ Data Insertion


In [157]:
# Insert books into the Cosmos DB container with embeddings
from collections import Counter

# Check whether at least one record exists before inserting
existing_items = next(iter(cosmos_container.read_all_items(max_item_count=1)), None)

# If container is not empty, skip insertion to avoid duplicates
if existing_items:
    print(
        f"⚠️ Container '{COSMOS_DB_CONTAINER}' already has data. Skipping insertion to avoid duplicates.")
else:
    # Limit to first 1000 books for demonstration
    books_batch = books_df[:1000]

    inserted_count = 0
    failure_reasons = Counter()

    for index, row in books_batch.iterrows():
        try:
            # Get the embedding for the book description
            book_title = row['title']
            book_description = row['description']
            embedding_context = f"{book_title}: {book_description}"

            book_embedding_response = await embedding_client.get_embeddings([embedding_context])
            book_embedding = book_embedding_response[0]

            # Create a document to insert into Cosmos DB
            document = {
                "id": str(row['isbn13']),  # Ensure the ID is a string
                "title": book_title,
                "authors": row['authors'],
                "categories": row['categories'],
                "description": book_description,
                "embeddings": book_embedding.vector
            }

            # Validate the document to ensure it can be serialized to JSON
            json.dumps(document)

            # Insert the document into the Cosmos DB container
            cosmos_container.upsert_item(document)
            inserted_count += 1

        except Exception as e:
            failure_reasons[type(e).__name__] += 1
            continue  # Skip to the next book in case of an error

    skipped_count = sum(failure_reasons.values())
    print(f"📊 Processed {len(books_batch)} books.")
    print(f"✅ Inserted {inserted_count} books into the Cosmos DB container with embeddings.")
    if skipped_count:
        reasons = ", ".join(f"{name} x{count}" for name, count in failure_reasons.most_common())
        print(f"⚠️ Skipped {skipped_count} books ({reasons}).")


⚠️ Container 'books' already has data. Skipping insertion to avoid duplicates.


#### 2.3.1 📚 Documents in the container
<img src="../assets/cosmosdb-documents.png" alt="Cosmos DB Documents" width="100%">


#### 2.3.2 💲 Embedding Token Usage
Tokens consumed generating embeddings for the `description` field across 1,000 records.

<img src="../assets/embedding-model.png" alt="Embedding Model" width="100%">


### 2.4 ✨ Cosmos DB Search Features
<img src="../assets/cosmosdb-features.png" alt="Cosmos DB Features" width="100%">


### 2.5 🔎 Keyword Search


ℹ️ **Keyword query structure**

```sql
SELECT TOP 5 c.id, c.title, c.authors, c.categories, c.description
FROM c
WHERE
    CONTAINS(c.title, @keyword, true)
    OR
    CONTAINS(c.description, @keyword, true)
```

- `CONTAINS` matches the term anywhere in the property — a substring match, not word-aware.
- The third argument `true` makes the comparison case-insensitive.
- `@keyword` is a bound parameter, passed separately so the term can't alter the query.
- This is pure lexical matching: a search for `science fiction` finds only that literal string.


In [158]:
# Keyword search sample query
search_keyword = "science fiction"

# CONTAINS with the case-insensitive flag matches the keyword anywhere in the title or description
keyword_query = """
SELECT TOP 5 c.id, c.title, c.authors, c.categories, c.description
FROM c
WHERE 
    CONTAINS(c.title, @keyword, true) 
    OR 
    CONTAINS(c.description, @keyword, true)
"""

In [159]:
# Run the keyword search query against the Cosmos DB container
keyword_results = list(cosmos_container.query_items(
    query=keyword_query,
    parameters=[{"name": "@keyword", "value": search_keyword}],
    enable_cross_partition_query=True
))

HIGHLIGHT = "\033[4;30;103m"  # underline + black text on a bright yellow background
RESET = "\033[0m"


def highlight(text, keyword):
    """Underline and highlight keyword matches using ANSI escape codes."""
    # Allow any whitespace between words so matches split across wrapped lines still highlight
    pattern = r"\s+".join(re.escape(word) for word in keyword.split())
    return re.sub(pattern, lambda m: f"{HIGHLIGHT}{m.group(0)}{RESET}", text, flags=re.IGNORECASE)


print(f"🔍 Keyword search for '{search_keyword}' returned {len(keyword_results)} results:\n")
for item in keyword_results:
    # Wrap before highlighting so the escape codes don't count towards the line width
    wrapped_description = textwrap.fill(
        item['description'], width=75, initial_indent='   ', subsequent_indent='   ')
    print(f"📘 {highlight(item['title'], search_keyword)} — {item['authors']}")
    print(f"{highlight(wrapped_description, search_keyword)}\n")


🔍 Keyword search for 'science fiction' returned 4 results:

📘 Gold — Isaac Asimov
   Gold is the final and crowning achievement of the fifty-year career of
   science fiction's transcendent genius, the world-famous author who
   defined the field of science fiction for its practitioners, its millions
   of readers, and the world at large. The first section contains stories
   that range from the humorous to the profound, at the heart of which is
   the title story, "Gold," a moving and revealing drama about a writer who
   gambles everything on a chance at immortality: a gamble Asimov himself
   made -- and won. The second section contains the grand master's
   ruminations on the SF genre itself. And the final section is comprised
   of Asimov's thoughts on the craft and writing of science fiction.

📘 Racso and the Rats of NIMH — Jane Leslie Conly
   ‘Racso, a brash and boastful little rodent, is making his way to Thorn
   Valley, determined to learn how to read and write and become a 

### 2.6 🔢 Vector Search


ℹ️ **Vector query structure**

```sql
SELECT TOP 5 c.title, c.authors, c.categories, c.description,
       VectorDistance(c.embeddings, @searchVector) AS similarity_score
FROM c
ORDER BY VectorDistance(c.embeddings, @searchVector)
```

- `VectorDistance` compares the stored vector against the query vector using the container's vector policy (cosine, 1536 dimensions).
- `@searchVector` is embedded by the **application** before the query — unlike PostgreSQL, where `azure_openai.create_embeddings` runs inside the database.
- `ORDER BY VectorDistance(...)` is what engages the vector index and turns this into a nearest-neighbour search.
- Repeating the expression in `SELECT` projects the distance so it can be shown as a similarity score.


In [160]:
# Vector (semantic) search sample query
search_phrase = "a lighthearted story about friendship and adventure"

# VectorDistance scores each document against the query vector; ORDER BY makes it a nearest-neighbour search
vector_query = """
SELECT TOP 5 c.title, c.authors, c.categories, c.description,
       VectorDistance(c.embeddings, @searchVector) AS similarity_score
FROM c
ORDER BY VectorDistance(c.embeddings, @searchVector)
"""

# Embed the search phrase with the same model used for the stored documents
search_embedding_response = await embedding_client.get_embeddings([search_phrase])
search_vector = search_embedding_response[0].vector

In [161]:
# Run the vector search query against the Cosmos DB container
vector_results = list(cosmos_container.query_items(
    query=vector_query,
    parameters=[{"name": "@searchVector", "value": search_vector}],
    enable_cross_partition_query=True
))

print(f"🧭 Vector search for '{search_phrase}' returned {len(vector_results)} results:\n")
for item in vector_results:
    score = f"{HIGHLIGHT}score: {item['similarity_score']:.4f}{RESET}"
    print(f"📘 {item['title']} — {item['authors']}  ({score})")
    print(textwrap.fill(item['description'], width=75,
          initial_indent='   ', subsequent_indent='   '), "\n")


🧭 Vector search for 'a lighthearted story about friendship and adventure' returned 5 results:

📘 Little House Friends — Heather Henson;Laura Ingalls Wilder  (score: 0.4327)
   Laura Ingalls shares adventures and good times with her friends while
   growing up on the western frontier. 

📘 The Giraffe and the Pelly and Me — Roald Dahl;Quentin Blake  (score: 0.4121)
   A Dahl story in which the giraffe, the pelican and the agile monkey set
   out to prove that they are the best window-cleaning company around. 

📘 The Illustrated Alchemist — Paulo Coelho;Alan R. Clarke;Moebius  (score: 0.4052)
   This fable aims teaches the reader to open their mind, listen to their
   heart and most importantly, follow their dreams. 

📘 Oliver and Albert, Friends Forever — Jean Van Leeuwen  (score: 0.3975)
   Oliver makes friends with Albert, the new boy in class, and they have
   fun together, playing kickball and collecting bugs. By the creators of
   Amanda Pig, Schoolgirl. Reprint. 

📘 Charlotte's Web

### 2.7 🔀 Hybrid Search


ℹ️ **Hybrid query structure**

```sql
SELECT TOP 5 c.title, c.authors, c.categories, c.description
FROM c
ORDER BY RANK RRF(
    VectorDistance(c.embeddings, @searchVector),
    FullTextScore(c.description, "friendship", "adventure")
)
```

- `ORDER BY RANK RRF(...)` is **built into Cosmos DB** — it fuses the rankings of both scoring functions for you.
- `FullTextScore` requires the container's full text policy and index, and only accepts **literal terms**, so the keywords are interpolated into the query text rather than bound as parameters.
- Adding a weight array as the last argument, e.g. `[2,1]`, biases the fusion — here the vector ranking counts twice as much as the keyword ranking.
- Compare with the PostgreSQL version further down, where the same fusion has to be written by hand with CTEs.


In [162]:
# Hybrid search: fuse keyword ranking and vector similarity with Reciprocal Rank Fusion (RRF)
hybrid_keywords = ["friendship", "adventure"]

# FullTextScore only accepts literal terms, so they go into the query text (json.dumps escapes them safely)
hybrid_terms = ", ".join(json.dumps(keyword) for keyword in hybrid_keywords)

# RRF (Reciprocal Rank Fusion) combines the ranks of 
# multiple scoring functions to produce a final ranking
hybrid_query = f"""
SELECT TOP 5 c.title, c.authors, c.categories, c.description
FROM c
ORDER BY RANK RRF(
    VectorDistance(c.embeddings, @searchVector),
    FullTextScore(c.description, {hybrid_terms})
)
"""

# Hybrid search with weighted ranking: 
# Give vector similarity twice the weight of keyword ranking
# [2,1] means the vector is weighted 2x, the keyword is weighted 1x
hybrid_weighted_query = f"""
SELECT TOP 5 c.title, c.authors, c.categories, c.description
FROM c
ORDER BY RANK RRF(
    VectorDistance(c.embeddings, @searchVector),    
    FullTextScore(c.description, {hybrid_terms}),
    [2,1]
)
"""

In [163]:
# Reuses the query vector from the vector search cell above
hybrid_results = list(cosmos_container.query_items(
    query=hybrid_query,
    parameters=[{"name": "@searchVector", "value": search_vector}],
    enable_cross_partition_query=True
))

print(f"🔀 Hybrid search for '{search_phrase}' + {hybrid_keywords} returned {len(hybrid_results)} results:\n")
for item in hybrid_results:
    print(f"📘 {item['title']} — {item['authors']}")
    print(textwrap.fill(item['description'], width=75,
          initial_indent='   ', subsequent_indent='   '), "\n")


🔀 Hybrid search for 'a lighthearted story about friendship and adventure' + ['friendship', 'adventure'] returned 5 results:

📘 Little House Friends — Heather Henson;Laura Ingalls Wilder
   Laura Ingalls shares adventures and good times with her friends while
   growing up on the western frontier. 

📘 Pippi Goes on Board — Astrid Lindgren;Louis S. Glanzman
   The further adventures of Pippi and her friends Tommy and Annika. 

📘 Of Mice and Men — John Steinbeck
   The tragic story of the friendship between two migrant workers, George
   and mentally retarded Lenny, and their dream of owning a farm 

📘 Captain Cat — Syd Hoff
   A patriotic feline, Captain Cat springs out of bed whenever the bugle
   sounds and he has more stripes than any of the soldiers. But most of
   all, this young recruit and his best friend Pete know what it really
   takes to make the army a home—friendship. ‘Hoff continues his string of
   hits.’ —BL. ‘Hoff has maintained his deft touch with a title that’s sure
  

### 2.8 🤖 Grounding an Agent with Search Results


In [164]:
# Create an agent that can answer questions about the books dataset 
# using the Foundry chat model
from agent_framework.foundry import FoundryChatClient
from agent_framework import Agent

FOUNDRY_CHAT_MODEL = os.getenv("FOUNDRY_CHAT_MODEL")
FOUNDRY_SERVICES_ENDPOINT = os.getenv("FOUNDRY_SERVICES_ENDPOINT")

chat_client = FoundryChatClient(
    project_endpoint=FOUNDRY_SERVICES_ENDPOINT,
    model=FOUNDRY_CHAT_MODEL,
    credential=credential)

agent = Agent(
        client=chat_client,
        name="LibraryAssistant",
        instructions="You're a friendly library assistant. You can answer questions about the books dataset, provide recommendations, and summarize book descriptions. Keep your answers brief and informative.",
    )

prompt = "Recommend a few books about friendship and adventure"

enriched_prompt = f"{prompt}\n\n{hybrid_results}"

result = await agent.run(enriched_prompt)

# print() writes plain text, so the model's markdown has to be displayed to render
display(Markdown(f"**🤖 Agent:**\n\n{result}"))


**🤖 Agent:**

Here are a few good picks about friendship and adventure:

- **Little House Friends** — Friendship and frontier adventures as Laura Ingalls grows up.
- **Pippi Goes on Board** — More fun adventures with Pippi and her friends Tommy and Annika.
- **Captain Cat** — A story that highlights friendship, teamwork, and life in the army.
- **Five Children and It** — Magical adventures and wishes that lead to lots of excitement.
- **Of Mice and Men** — A more serious story about friendship and shared dreams.

If you want, I can also narrow these down to **younger readers** or **more adventurous** picks.

### 2.9 📚 References

**Concepts**

- [Vector search in Azure Cosmos DB for NoSQL](https://learn.microsoft.com/azure/cosmos-db/vector-search) — container vector policies, index types (`flat`, `quantizedFlat`, `diskANN`), and dimension limits.
- [Full-text search in Azure Cosmos DB for NoSQL](https://learn.microsoft.com/azure/cosmos-db/gen-ai/full-text-search) — full-text policy, full-text index, and BM25 scoring.
- [Hybrid search in Azure Cosmos DB for NoSQL](https://learn.microsoft.com/azure/cosmos-db/gen-ai/hybrid-search) — combining vector and full-text scoring with Reciprocal Rank Fusion.
- [Retrieval Augmented Generation (RAG)](https://learn.microsoft.com/azure/cosmos-db/gen-ai/rag) — grounding a model on search results.
- [Manage indexing policies](https://learn.microsoft.com/azure/cosmos-db/how-to-manage-indexing-policy) — vector and full-text index policy examples.

**Query language reference**

- [`VECTORDISTANCE`](https://learn.microsoft.com/cosmos-db/query/vectordistance) — similarity score between two vectors.
- [`FULLTEXTSCORE`](https://learn.microsoft.com/cosmos-db/query/fulltextscore) — BM25 relevance score, usable only in `ORDER BY RANK`.
- [`RRF`](https://learn.microsoft.com/cosmos-db/query/rrf) — fuses two or more scoring functions, with optional weights.


---
---

---

## 3. 🐘 Azure Database for PostgreSQL


```mermaid
%%{init: {"themeVariables": {"fontSize": "20px"}}}%%
flowchart LR
    A([3.1 Setup]) --> B([3.2 Insert data])
    B --> C([3.3 Embeddings])
    C --> D([3.4 Vector search])
    C --> E([3.5 Hybrid search])
    C --> F([3.6 Graph search])
```


### 3.1 🛠️ Setup and Configuration


In [165]:
# Prepare the PostgreSQL configurations
from azure.mgmt.postgresqlflexibleservers import PostgreSQLManagementClient
from azure.mgmt.postgresqlflexibleservers.models import Database

RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID")
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_TABLE = os.getenv("POSTGRES_TABLE")

# The server name is the first label of the host (e.g. my-server.postgres.database.azure.com)
POSTGRES_SERVER = POSTGRES_HOST.split('.')[0]

# This is an ARM control-plane client, so it must talk to management.azure.com (the SDK default),
# not to the server host on port 5432, which speaks the Postgres wire protocol instead of HTTPS.
postgres_client = PostgreSQLManagementClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID
)

# Create the PostgreSQL database if it doesn't exist
response = postgres_client.databases.begin_create(
    resource_group_name=RESOURCE_GROUP,
    server_name=POSTGRES_SERVER,
    database_name=POSTGRES_DB,
    parameters=Database(charset="utf8", collation="en_US.utf8"),
).result()  # Wait for the operation to complete

print(f"✅ PostgreSQL database '{POSTGRES_DB}' created or already exists.")

✅ PostgreSQL database 'books-db' created or already exists.


In [166]:
# Check PostgreSQL extensions to ensure the required extensions are available
extensions_config = postgres_client.configurations.get(
    resource_group_name=RESOURCE_GROUP,
    server_name=POSTGRES_SERVER,
    configuration_name="azure.extensions",
)

# List the allow-listed extensions from the configuration
allow_listed = [name for name in (
    extensions_config.value or "").split(",") if name]

print(f"🧩 Enabled Azure extensions on the PostgreSQL server ({len(allow_listed)}):")
for name in sorted(allow_listed):
    print(f"   • {name}")

🧩 Enabled Azure extensions on the PostgreSQL server (4):
   • age
   • azure_ai
   • pg_diskann
   • vector


#### 3.1.1 🧩 Allow-listed Azure Extensions
<img src="../assets/postgres-az-extensions.png" alt="PostgreSQL Azure Extensions" width="100%">


### 3.2 ✍️ Data Insertion


In [182]:
# Create a Postgres table for the books dataset if it doesn't exist
import psycopg2
from psycopg2 import sql

# Entra ID auth: the access token is the password, and the login name is the signed-in principal
pg_token = credential.get_token(
    "https://ossrdbms-aad.database.windows.net/.default").token

# Connect to the PostgreSQL database using psycopg2 with Entra ID authentication
postgres_connection = psycopg2.connect(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    dbname=POSTGRES_DB,
    user=claims["upn"],
    password=pg_token,
    sslmode="require",
    # libpq 18 defaults to gssencmode='prefer'; the Azure gateway drops the GSSENCRequest packet
    gssencmode="disable",
)
postgres_connection.autocommit = True

# Create the table if it doesn't exist
with postgres_connection.cursor() as cursor:
    # Identifier() quotes the env-provided table name so it can't inject SQL
    cursor.execute(sql.SQL("""
        CREATE TABLE IF NOT EXISTS {table} (
            id          TEXT PRIMARY KEY,
            title       TEXT,
            authors     TEXT,
            categories  TEXT,
            description TEXT
        );
    """).format(table=sql.Identifier(POSTGRES_TABLE)))

print(f"✅ Table '{POSTGRES_TABLE}' created or already exists in '{POSTGRES_DB}'.")


✅ Table 'books' created or already exists in 'books-db'.


In [168]:
# Insert books into the PostgreSQL table
inserted_count = 0
books_batch = books_df[:1000]  # Limit to first 1000 books for demonstration

# Check whether at least one record exists before inserting
with postgres_connection.cursor() as cursor:
    cursor.execute(sql.SQL("SELECT 1 FROM {table} LIMIT 1;").format(
        table=sql.Identifier(POSTGRES_TABLE)))
    existing_record = cursor.fetchone()
    
    if existing_record:
        print(
            f"⚠️ Table '{POSTGRES_TABLE}' already has data. Skipping insertion to avoid duplicates.")
    else:
        for index, row in books_batch.iterrows():
            try:
                with postgres_connection.cursor() as cursor:
                    cursor.execute(sql.SQL("""
                        INSERT INTO {table} (id, title, authors, categories, description)
                        VALUES (%s, %s, %s, %s, %s)
                        ON CONFLICT (id) DO NOTHING;
                    """).format(table=sql.Identifier(POSTGRES_TABLE)),
                        (str(row['isbn13']), 
                         row['title'], 
                         row['authors'], 
                         row['categories'], 
                         row['description']))
                    inserted_count += 1
            except Exception as e:
                print(f"⚠️ Failed to insert book '{row['title']}': {e}")
                continue  # Skip to the next book in case of an error

        print(f"✅ Inserted {inserted_count} books into the PostgreSQL table.")


⚠️ Table 'books' already has data. Skipping insertion to avoid duplicates.


### 3.3 🧮 Generate Embeddings


In [183]:
# Configure the PostgreSQL server to Foundry for generating embeddings
# The azure_ai extension calls the OpenAI-form endpoint, not the .services.ai.azure.com one
FOUNDRY_OPENAI_ENDPOINT = os.getenv("FOUNDRY_OPENAI_ENDPOINT")
FOUNDRY_EMBEDDING_SMALL_MODEL = os.getenv("FOUNDRY_EMBEDDING_SMALL_MODEL")

with postgres_connection.cursor() as cursor:
    cursor.execute("CREATE EXTENSION IF NOT EXISTS azure_ai;")

    # Managed identity keeps an API key out of the extension's settings table
    cursor.execute(
        "SELECT azure_ai.set_setting('azure_openai.auth_type', 'managed-identity');")
    cursor.execute(
        "SELECT azure_ai.set_setting('azure_openai.endpoint', %s);", (FOUNDRY_OPENAI_ENDPOINT,))

    # Verify the settings were applied correctly
    cursor.execute("""
        SELECT azure_ai.get_setting('azure_openai.auth_type'),
               azure_ai.get_setting('azure_openai.endpoint');
    """)
    auth_type, endpoint = cursor.fetchone()

print("✅ PostgreSQL server configured for Foundry embeddings.")
print(f"🔐 Auth type: {auth_type}")
# Compared rather than printed, so the resource name stays out of the committed notebook
print(f"🌐 Endpoint:  {'matches FOUNDRY_OPENAI_ENDPOINT' if endpoint == FOUNDRY_OPENAI_ENDPOINT else '⚠️ differs from .env'}")
print(f"🤖 Embeddings model: {FOUNDRY_EMBEDDING_SMALL_MODEL}")


✅ PostgreSQL server configured for Foundry embeddings.
🔐 Auth type: managed-identity
🌐 Endpoint:  matches FOUNDRY_OPENAI_ENDPOINT
🤖 Embeddings model: text-embedding-3-small


#### 3.3.1 ✨ PostgreSQL + Azure OpenAI Integration
<img src="../assets/postgres-aoai.png" alt="Postgres and Azure OpenAI" width="100%">


In [170]:
# Add a vector column to the PostgreSQL table for storing embeddings
import time

# DiskANN caps out at 2000 dimensions, so this demo uses the 1536-dimension small model
FOUNDRY_EMBEDDING_SMALL_MODEL = os.getenv("FOUNDRY_EMBEDDING_SMALL_MODEL")

EMBEDDING_DIMENSIONS = 1536    # text-embedding-3-small
EMBEDDING_CHUNK_SIZE = 50      # rows per statement, so a throttled call costs one chunk instead of the whole run
EMBEDDING_PAUSE_SECONDS = 60   # let the per-minute quota refill between chunks

with postgres_connection.cursor() as cursor:

    # pgvector provides the `vector` type; pg_diskann provides the ANN index
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cursor.execute("CREATE EXTENSION IF NOT EXISTS pg_diskann CASCADE;")

    cursor.execute(sql.SQL("""
        ALTER TABLE {table}
        ADD COLUMN IF NOT EXISTS embeddings vector({dims});
    """).format(table=sql.Identifier(POSTGRES_TABLE),
                dims=sql.Literal(EMBEDDING_DIMENSIONS)))

    # Resize the column if it was created for a different embedding model (requires the column to be NULL)
    cursor.execute(sql.SQL("""
        ALTER TABLE {table}
        ALTER COLUMN embeddings TYPE vector({dims});
    """).format(table=sql.Identifier(POSTGRES_TABLE),
                dims=sql.Literal(EMBEDDING_DIMENSIONS)))

    cursor.execute(sql.SQL("""
        SELECT COUNT(*) FROM {table}
        WHERE embeddings IS NULL AND description IS NOT NULL;
    """).format(table=sql.Identifier(POSTGRES_TABLE)))
    pending_count = cursor.fetchone()[0]
    print(f"🧮 {pending_count} records need embeddings.")

    # Targeting only NULL embeddings makes the cell resumable: re-run it after a rate-limit failure
    # and it picks up where it stopped. autocommit means each chunk is durable on its own.
    embedded_count = 0
    backoff_seconds = EMBEDDING_PAUSE_SECONDS

    while True:
        try:
            cursor.execute(sql.SQL("""
                UPDATE {table}
                SET embeddings = azure_openai.create_embeddings(
                        %s, description,
                        max_attempts => 5,
                        retry_delay_ms => 10000
                    )::vector
                WHERE id IN (
                    SELECT id FROM {table}
                    WHERE embeddings IS NULL AND description IS NOT NULL
                    LIMIT %s
                );
            """).format(table=sql.Identifier(POSTGRES_TABLE)),
                (FOUNDRY_EMBEDDING_SMALL_MODEL, EMBEDDING_CHUNK_SIZE))
        except psycopg2.Error as error:
            if "RateLimitReached" not in str(error):
                raise
            # The quota is per minute, so back off further each time instead of retrying at a fixed delay
            print(f"   ⏳ throttled — waiting {backoff_seconds}s before retrying this chunk")
            time.sleep(backoff_seconds)
            backoff_seconds = min(backoff_seconds * 2, 120)
            continue

        if cursor.rowcount == 0:
            break

        embedded_count += cursor.rowcount
        backoff_seconds = EMBEDDING_PAUSE_SECONDS
        print(f"   … {embedded_count}/{pending_count} embedded")
        time.sleep(EMBEDDING_PAUSE_SECONDS)

    print(f"✅ Embeddings generated for '{POSTGRES_TABLE}' using '{FOUNDRY_EMBEDDING_SMALL_MODEL}'.")

    # Build the index after the column is populated so DiskANN sees the real data distribution
    cursor.execute(sql.SQL("""
        CREATE INDEX IF NOT EXISTS {index} ON {table}
        USING diskann (embeddings vector_cosine_ops);
    """).format(index=sql.Identifier(f"{POSTGRES_TABLE}_embeddings_diskann"),
                table=sql.Identifier(POSTGRES_TABLE)))

    print(f"✅ DiskANN index ready on '{POSTGRES_TABLE}.embeddings' ({EMBEDDING_DIMENSIONS} dimensions).")


🧮 0 records need embeddings.
✅ Embeddings generated for 'books' using 'text-embedding-3-small'.
✅ DiskANN index ready on 'books.embeddings' (1536 dimensions).


#### 3.3.2 📚 Records in the table
<img src="../assets/postgres-records.png" alt="PostgreSQL List of Records" width="100%">


### 3.4 🔢 Vector Search


ℹ️ **Vector query structure**

```sql
SELECT title, authors, categories, description
FROM books
ORDER BY embeddings <=> azure_openai.create_embeddings('embeddings-deployment', 'search phrase')::vector
LIMIT 5;
```

- `azure_openai.create_embeddings(...)` embeds the search phrase **inside the database** — no round-trip through the app.
- `::vector` casts the returned `real[]` into the `vector` type stored in the column.
- `<=>` is cosine distance, matching the `vector_cosine_ops` DiskANN index.
- `ORDER BY` + `LIMIT` is what turns a distance calculation into a nearest-neighbour search.


In [171]:
# Vector (semantic) search entirely inside PostgreSQL
postgres_search_phrase = "a lighthearted story about friendship and adventure"

# <=> is pgvector's cosine distance operator, and matches the vector_cosine_ops index
# create_embeddings() sits in ORDER BY only, so the phrase is embedded once per query
with postgres_connection.cursor() as cursor:
    
    # Use the same embedding model as the one used for the stored documents
    cursor.execute(sql.SQL("""
        SELECT title, authors, categories, description
        FROM {table}
        ORDER BY embeddings <=> azure_openai.create_embeddings(%s, %s)::vector
        LIMIT 5;
    """).format(table=sql.Identifier(POSTGRES_TABLE)),
        (FOUNDRY_EMBEDDING_SMALL_MODEL, postgres_search_phrase))

    # Fetch the results from the query
    postgres_vector_results = cursor.fetchall()

print(f"🧭 Vector search for '{postgres_search_phrase}' returned {len(postgres_vector_results)} results:\n")
for title, authors, categories, description in postgres_vector_results:
    print(f"📘 {title} — {authors}")
    print(textwrap.fill(description, width=75,
          initial_indent='   ', subsequent_indent='   '), "\n")


🧭 Vector search for 'a lighthearted story about friendship and adventure' returned 5 results:

📘 The Canterbury Tales — Geoffrey Chaucer
   A retelling of the medieval poem about a group of travelers on a
   pilgrimage to Canterbury and the tales they tell each other 

📘 Five Children and it — Edith Nesbit
   A series of phenomenal adventures follow when young Anthea discovers a
   sand-fairy who can grant wishes. 

📘 Of Mice and Men — John Steinbeck
   The tragic story of the friendship between two migrant workers, George
   and mentally retarded Lenny, and their dream of owning a farm 

📘 Westmark — Lloyd Alexander
   Theo, a boy fleeing from criminal charges, falls in with a charlatan,
   his dwarf attendant, and an urchin girl; travels with them about the
   kingdom of Westmark; and ultimately arrives at the palace where the king
   is grieving over the loss of his daughter. An ALA Notable Book. Reissue. 

📘 The Log from the Sea of Cortez — John Steinbeck
   This light-hearted jour

### 3.5 🔀 Hybrid Search


ℹ️ **Hybrid query structure**

```sql
WITH query_embedding AS (
    SELECT azure_openai.create_embeddings('embeddings-deployment', 'search phrase')::vector AS embedding
),
semantic AS (          -- rank by meaning
    SELECT b.id, ROW_NUMBER() OVER (ORDER BY b.embeddings <=> q.embedding) AS rank
    FROM books b, query_embedding q
    ORDER BY b.embeddings <=> q.embedding
    LIMIT 50
),
keyword AS (           -- rank by words
    SELECT b.id, ROW_NUMBER() OVER (ORDER BY ts_rank_cd(to_tsvector(...),websearch_to_tsquery(...)) DESC) AS rank
    FROM books b
    WHERE to_tsvector(...) @@ websearch_to_tsquery(...)
    LIMIT 50
)
SELECT b.title, ..., 1.0 / (60 + s.rank) + 1.0 / (60 + k.rank) AS rrf_score
FROM semantic s
FULL OUTER JOIN keyword k ON s.id = k.id
JOIN books b ON b.id = coalesce(s.id, k.id)
ORDER BY rrf_score DESC
LIMIT 5;
```

- The embedding is generated **once** in a CTE and reused by the semantic branch.
- Each strategy produces its own ranked candidate list, then Reciprocal Rank Fusion sums `1 / (k + rank)` from both.
- `FULL OUTER JOIN` keeps books that only one strategy found; the missing side contributes `0`.
- Unlike Cosmos DB's built-in `RRF()`, PostgreSQL has no fusion operator — it is written out explicitly.


In [172]:
# Hybrid search: fuse full-text ranking and vector similarity with Reciprocal Rank Fusion (RRF)
postgres_hybrid_keywords = "friendship or adventure"

RRF_K = 60            # standard smoothing constant, dampens the influence of top ranks
RRF_CANDIDATES = 50   # how many rows each strategy contributes before fusion

# Postgres has no built-in RRF, so each strategy is ranked in its own CTE and the
# reciprocal ranks are summed; FULL OUTER JOIN keeps rows found by only one of them.
with postgres_connection.cursor() as cursor:
    
    # Use the same embedding model as the one used for the stored documents    
    cursor.execute(sql.SQL("""
        WITH query_embedding AS (
            SELECT azure_openai.create_embeddings(%s, %s)::vector AS embedding
        ),
        semantic AS (
            SELECT b.id, ROW_NUMBER() OVER (ORDER BY b.embeddings <=> q.embedding) AS rank
            FROM {table} b, query_embedding q
            WHERE b.embeddings IS NOT NULL
            ORDER BY b.embeddings <=> q.embedding
            LIMIT %s
        ),
        keyword AS (
            SELECT b.id, ROW_NUMBER() OVER (
                       ORDER BY ts_rank_cd(
                           to_tsvector('english', coalesce(b.title, '') || ' ' || coalesce(b.description, '')),
                           websearch_to_tsquery('english', %s)) DESC) AS rank
            FROM {table} b
            WHERE to_tsvector('english', coalesce(b.title, '') || ' ' || coalesce(b.description, ''))
                  @@ websearch_to_tsquery('english', %s)
            LIMIT %s
        )
        SELECT b.title, b.authors, b.categories, b.description,
               coalesce(1.0 / (%s + s.rank), 0.0) + coalesce(1.0 / (%s + k.rank), 0.0) AS rrf_score
        FROM semantic s
        FULL OUTER JOIN keyword k ON s.id = k.id
        JOIN {table} b ON b.id = coalesce(s.id, k.id)
        ORDER BY rrf_score DESC
        LIMIT 5;
    """).format(table=sql.Identifier(POSTGRES_TABLE)),
        (FOUNDRY_EMBEDDING_SMALL_MODEL, postgres_search_phrase, RRF_CANDIDATES,
         postgres_hybrid_keywords, postgres_hybrid_keywords, RRF_CANDIDATES,
         RRF_K, RRF_K))

    # Fetch the results from the query
    postgres_hybrid_results = cursor.fetchall()

print(f"🔀 Hybrid search for '{postgres_search_phrase}' + '{postgres_hybrid_keywords}' "
      f"returned {len(postgres_hybrid_results)} results:\n")
for title, authors, categories, description, rrf_score in postgres_hybrid_results:
    score = f"{HIGHLIGHT}RRF: {rrf_score:.4f}{RESET}"
    print(f"📘 {title} — {authors}  ({score})")
    print(textwrap.fill(description, width=75,
          initial_indent='   ', subsequent_indent='   '), "\n")


🔀 Hybrid search for 'a lighthearted story about friendship and adventure' + 'friendship or adventure' returned 5 results:

📘 Five Children and it — Edith Nesbit  (RRF: 0.0306)
   A series of phenomenal adventures follow when young Anthea discovers a
   sand-fairy who can grant wishes. 

📘 The Valkyries — Paulo Coelho  (RRF: 0.0302)
   A Magical Tale About Forgiving Our Past and Believing in Our Future The
   enchanting, true story of The Valkyries begins in Rio de Janeiro when
   author Paulo Coelho gives his mysterious master J., the only manuscript
   for his book The Alchemist. Haunted by a devastating curse, Coelho
   confesses to J., "I′ve seen my dreams fall apart just when I seemed
   about to achieve them." In response, J. gives Coelho a daunting task: He
   must find and speak with his guardian angel. "The curse can be broken,"
   he replies, "if you complete the task." Rising to the challenge, Paulo
   and his wife, Cristina, drop everything, pack their bags, and take off
   

### 3.6 🕸️ Graph Search

Every search so far treats a book as an independent row. A graph asks a different question:
**how are these books connected, and what can you reach by following those connections?**

The `age` extension (Apache AGE) brings the openCypher query language into PostgreSQL, so the graph
lives in the same database as the rows *and* the vectors — no separate graph service, no ETL.

> This section has no Cosmos DB counterpart. Cosmos DB's graph story is the Gremlin API, which is a
> different account kind and can't be added to the NoSQL account used earlier.


#### 3.6.1 🧭 The Graph Model


```mermaid
%%{init: {"themeVariables": {"fontSize": "20px"}}}%%
flowchart LR
    A([Author]) -->|WROTE| B([Book])
    B -->|IN_CATEGORY| C([Category])
    B -->|PUBLISHED_IN| D([Decade])
    B -.->|SIMILAR_TO score| B2([Book])
```

Nothing here is invented — every node and edge is projected from columns already in the `books` table:

| Edge | Built from | |
|---|---|---|
| `WROTE` | `authors`, split on `;` | one row can name several authors |
| `IN_CATEGORY` | `categories` | |
| `PUBLISHED_IN` | `published_year`, bucketed into decades | |
| `SIMILAR_TO {score}` | top-_k_ nearest neighbours over `embeddings` | **derived from the real embeddings**, not synthetic |

`SIMILAR_TO` is the interesting one. It is the only Book→Book edge, it is what makes the graph more
than a star of lookup tables, and it is the one relationship no `JOIN` could ever produce — it only
exists because the vectors were computed first.

> ⚙️ **Server prerequisite:** `age` must appear in **both** the `azure.extensions` **and**
> `shared_preload_libraries` server parameters. Changing `shared_preload_libraries` restarts the
> server. Because Azure preloads the library for you, `LOAD 'age';` is unnecessary and raises a
> privilege error — unlike a self-hosted AGE install.


In [173]:
# Bring in three CSV columns the original import skipped; the graph needs them for Decade nodes
from psycopg2.extras import execute_values

GRAPH_BOOK_LIMIT = 300   # a smaller graph builds and traverses fast enough to demo live

with postgres_connection.cursor() as cursor:
    cursor.execute(sql.SQL("""
        ALTER TABLE {table}
            ADD COLUMN IF NOT EXISTS published_year INT,
            ADD COLUMN IF NOT EXISTS average_rating REAL,
            ADD COLUMN IF NOT EXISTS ratings_count  INT;
    """).format(table=sql.Identifier(POSTGRES_TABLE)))

    enrichment = [
        (str(row.isbn13),
         None if pd.isna(row.published_year) else int(row.published_year),
         None if pd.isna(row.average_rating) else float(row.average_rating),
         None if pd.isna(row.ratings_count) else int(row.ratings_count))
        for row in books_df.itertuples()
    ]

    # One UPDATE ... FROM (VALUES ...) instead of a thousand round trips
    # execute_values() needs plain SQL text, so the composed statement is rendered first
    execute_values(cursor, sql.SQL("""
        UPDATE {table} AS t
        SET published_year = v.year::int,
            average_rating = v.rating::real,
            ratings_count  = v.cnt::int
        FROM (VALUES %s) AS v(id, year, rating, cnt)
        WHERE t.id = v.id::text;
    """).format(table=sql.Identifier(POSTGRES_TABLE)).as_string(cursor), enrichment)

    cursor.execute(sql.SQL("SELECT count(published_year), min(published_year), max(published_year) FROM {table};")
                   .format(table=sql.Identifier(POSTGRES_TABLE)))
    enriched, earliest, latest = cursor.fetchone()

print(f"✅ Enriched {enriched} rows — publication years span {earliest} to {latest}.")


✅ Enriched 1000 rows — publication years span 1940 to 2010.


In [174]:
# Enable Apache AGE and create an empty graph
import json

GRAPH_NAME = "books_graph"


def run_cypher(cursor, cypher_query, params=None, returns=("result",)):
    # AGE rejects anything but a real bind parameter as cypher()'s third argument, and psycopg2
    # interpolates client-side into a literal -- so PREPARE supplies the $1 that AGE insists on.
    # Output names are quoted because Cypher aliases like "similar" or "count" are SQL reserved words.
    columns = ", ".join(f'"{name}" agtype' for name in returns)
    cursor.execute("DEALLOCATE ALL;")
    cursor.execute(f"""
        PREPARE age_query(agtype) AS
        SELECT * FROM cypher('{GRAPH_NAME}', $cypher${cypher_query}$cypher$, $1) AS ({columns});
    """)
    cursor.execute("EXECUTE age_query(%s);", (json.dumps(params or {}),))
    return cursor.fetchall()


def agval(value):
    return json.loads(value) if isinstance(value, str) else value   # agtype scalars arrive JSON-quoted


with postgres_connection.cursor() as cursor:
    cursor.execute("CREATE EXTENSION IF NOT EXISTS age CASCADE;")

    # ag_catalog holds create_graph(), cypher() and the agtype operators
    # No LOAD 'age' here: Azure preloads the library, and calling LOAD raises a privilege error
    cursor.execute('SET search_path = ag_catalog, "$user", public;')

    cursor.execute("SELECT count(*) FROM ag_graph WHERE name = %s;", (GRAPH_NAME,))
    if cursor.fetchone()[0]:
        print(f"⚠️ Graph '{GRAPH_NAME}' already exists — skipping creation to avoid duplicate nodes.")
        print(f"   To rebuild from scratch: SELECT drop_graph('{GRAPH_NAME}', true);")
    else:
        cursor.execute("SELECT create_graph(%s);", (GRAPH_NAME,))
        print(f"✅ Graph '{GRAPH_NAME}' created.")

    cursor.execute("SELECT extversion FROM pg_extension WHERE extname = 'age';")
    print(f"🕸️ Apache AGE version: {cursor.fetchone()[0]}")


⚠️ Graph 'books_graph' already exists — skipping creation to avoid duplicate nodes.
   To rebuild from scratch: SELECT drop_graph('books_graph', true);
🕸️ Apache AGE version: 1.6.0


In [175]:
# Project the relational rows into Book / Author / Category / Decade nodes and their edges
with postgres_connection.cursor() as cursor:
    cursor.execute(sql.SQL("""
        SELECT id, title, authors, categories, published_year, average_rating
        FROM {table}
        WHERE embeddings IS NOT NULL
        ORDER BY id
        LIMIT %s;
    """).format(table=sql.Identifier(POSTGRES_TABLE)), (GRAPH_BOOK_LIMIT,))
    scoped_books = cursor.fetchall()

# Per edge type: the label it connects to, and the pattern that orients it
EDGE_TYPES = {
    "WROTE":        ("Author",   "(n)-[:WROTE]->(b)"),
    "IN_CATEGORY":  ("Category", "(b)-[:IN_CATEGORY]->(n)"),
    "PUBLISHED_IN": ("Decade",   "(b)-[:PUBLISHED_IN]->(n)"),
}

books = []
names = {label: set() for label, _ in EDGE_TYPES.values()}
links = {edge: [] for edge in EDGE_TYPES}

for book_id, title, authors, categories, year, rating in scoped_books:
    books.append({"id": book_id, "title": title, "year": year, "rating": rating})

    # authors and categories are semicolon-separated, so one row can fan out to several nodes
    for edge, values in (("WROTE", (authors or "").split(";")),
                         ("IN_CATEGORY", (categories or "").split(";")),
                         ("PUBLISHED_IN", [f"{year // 10 * 10}s"] if year else [])):
        for name in values:
            if name := name.strip():
                names[EDGE_TYPES[edge][0]].add(name)
                links[edge].append({"name": name, "book": book_id})


def count_in_graph(cursor, pattern, variable):
    return agval(run_cypher(cursor, f"MATCH {pattern} RETURN count({variable})", returns=("total",))[0][0])


with postgres_connection.cursor() as cursor:
    # CREATE is not idempotent: without this wipe, re-running the cell multiplies every node and edge
    run_cypher(cursor, "MATCH (n) DETACH DELETE n")

    # disp_label is what the VS Code PostgreSQL extension shows on a node; without it you get raw IDs
    run_cypher(cursor, """
        UNWIND $rows AS r
        CREATE (:Book {id: r.id, title: r.title, year: r.year, rating: r.rating, disp_label: r.title})
    """, {"rows": books})

    # MERGE deduplicates: hundreds of books collapse onto far fewer authors, categories and decades
    # name and disp_label are always equal here, so carrying both in the pattern is still unique per name
    for label, values in names.items():
        run_cypher(cursor, f"UNWIND $rows AS r MERGE (:{label} {{name: r, disp_label: r}})",
                   {"rows": sorted(values)})

    for edge, (label, pattern) in EDGE_TYPES.items():
        run_cypher(cursor, f"""
            UNWIND $rows AS r
            MATCH (b:Book {{id: r.book}}), (n:{label} {{name: r.name}})
            CREATE {pattern}
        """, {"rows": links[edge]})

    # Counted back out of the graph rather than from the lists above, so drift shows up here
    node_counts = {label: count_in_graph(cursor, f"(n:{label})", "n") for label in ("Book", *names)}
    edge_counts = {edge: count_in_graph(cursor, f"()-[r:{edge}]->()", "r") for edge in EDGE_TYPES}

print("✅ Nodes: " + " · ".join(f"{v} {k}" for k, v in node_counts.items()))
print("✅ Edges: " + " · ".join(f"{v} {k}" for k, v in edge_counts.items()))
print("⚠️ SIMILAR_TO edges were wiped too — re-run the next cell to rebuild them.")


✅ Nodes: 300 Book · 196 Author · 84 Category · 6 Decade
✅ Edges: 324 WROTE · 300 IN_CATEGORY · 300 PUBLISHED_IN
⚠️ SIMILAR_TO edges were wiped too — re-run the next cell to rebuild them.


In [176]:
# Turn vector neighbours into graph edges -- the one relationship no JOIN could produce
SIMILAR_TO_K = 5   # neighbours kept per book

with postgres_connection.cursor() as cursor:
    # LATERAL runs the nearest-neighbour search once per book, reusing the same DiskANN index
    cursor.execute(sql.SQL("""
        WITH scoped AS (
            SELECT id, embeddings FROM {table}
            WHERE embeddings IS NOT NULL ORDER BY id LIMIT %s
        )
        SELECT s.id, n.id, round((1 - (s.embeddings <=> n.embeddings))::numeric, 4)
        FROM scoped s
        CROSS JOIN LATERAL (
            SELECT o.id, o.embeddings FROM scoped o
            WHERE o.id <> s.id
            ORDER BY s.embeddings <=> o.embeddings
            LIMIT %s
        ) AS n;
    """).format(table=sql.Identifier(POSTGRES_TABLE)), (GRAPH_BOOK_LIMIT, SIMILAR_TO_K))

    # 1 - cosine distance, so a higher score means a closer book
    neighbours = [{"src": src, "dst": dst, "score": float(score)} for src, dst, score in cursor.fetchall()]

    # Same reason as the previous cell: clear before CREATE so the cell can be re-run on its own
    run_cypher(cursor, "MATCH ()-[r:SIMILAR_TO]->() DELETE r")

    run_cypher(cursor, """
        UNWIND $rows AS r
        MATCH (a:Book {id: r.src}), (b:Book {id: r.dst})
        CREATE (a)-[:SIMILAR_TO {score: r.score}]->(b)
    """, {"rows": neighbours})

    created = count_in_graph(cursor, "()-[r:SIMILAR_TO]->()", "r")

print(f"✅ {created} SIMILAR_TO edges in the graph ({SIMILAR_TO_K} per book, {len(neighbours)} expected).")


✅ 1500 SIMILAR_TO edges in the graph (5 per book, 1500 expected).


ℹ️ **Graph query structure**

```sql
PREPARE two_hops(agtype) AS
SELECT * FROM cypher('books_graph', $$
    MATCH (seed:Book {title: $title})<-[:WROTE]-(a:Author)-[:WROTE]->(other:Book)
    WHERE other <> seed
    RETURN a.name, other.title
$$, $1) AS ("author" agtype, "title" agtype);

EXECUTE two_hops('{"title": "Gilead"}');
```

- `cypher(graph, query, params)` is an ordinary set-returning **SQL function** — a graph traversal can sit inside any `SELECT`, `JOIN` or CTE.
- The Cypher text must be a literal, because AGE parses it while planning. Values go in the third argument as `agtype` instead, which is where `$title` comes from.
- That third argument **must be a bind parameter**, not a literal — AGE checks the parse tree and rejects a constant with *"third argument of cypher function must be a parameter"*. Since psycopg2 substitutes parameters client-side, the only way to hand AGE a real parameter is to `PREPARE` the statement with `$1` and pass the JSON through `EXECUTE`.
- `AS (...)` is **mandatory**: PostgreSQL needs the output columns declared up front, one per item in `RETURN`.
- Those output names are ordinary **SQL identifiers**, so quote them. Natural Cypher aliases such as `similar`, `count`, `order` or `type` are PostgreSQL reserved words and fail with a bare `syntax error` otherwise.
- `(a)-[:WROTE]->(b)` is the pattern; arrows carry direction, and `<-` simply reads the same edge backwards.
- Results come back as `agtype`, so strings arrive JSON-quoted — hence the small `agval()` helper.
- `ag_catalog` must be on the `search_path`. That is set once per **connection**, not per database.

> 💡 The [PostgreSQL extension for VS Code](https://learn.microsoft.com/azure/postgresql/development/vs-code-extension/postgresql-extension-overview) renders Cypher results as an interactive node-edge graph — but only when the query returns whole nodes and edges (`RETURN a, r, b`) rather than scalar properties.


In [177]:
# Traversal: find the hubs, then walk two hops out from one of their books
with postgres_connection.cursor() as cursor:
    # count(DISTINCT b) so an author is credited once per book, however many paths reach it
    hub_authors = run_cypher(cursor, """
        MATCH (a:Author)-[:WROTE]->(b:Book)
        WITH a.name AS author, count(DISTINCT b) AS books
        RETURN author, books
        ORDER BY books DESC, author
        LIMIT 10
    """, returns=("author", "books"))

    # Seed from the busiest author so the traversal is guaranteed to return something
    graph_seed_title = agval(run_cypher(cursor, """
        MATCH (:Author {name: $author})-[:WROTE]->(b:Book)
        RETURN b.title ORDER BY b.title LIMIT 1
    """, {"author": agval(hub_authors[0][0])}, returns=("title",))[0][0])

    # Book -> Author -> Book: the classic two-hop that a graph makes trivial
    shared_author_books = run_cypher(cursor, """
        MATCH (seed:Book {title: $title})<-[:WROTE]-(a:Author)-[:WROTE]->(other:Book)
        WHERE other.title <> $title
        RETURN DISTINCT a.name, other.title, other.year
        ORDER BY other.year
    """, {"title": graph_seed_title}, returns=("author", "title", "year"))

print("🏛️ Most connected authors in the graph:\n")
for author, books in hub_authors:
    print(f"   {agval(books):>3} books — {agval(author)}")

print(f"\n🔗 Two hops from '{graph_seed_title}' (book → author → book):\n")
for author, title, year in shared_author_books:
    print(f"📘 {agval(title)} ({agval(year)}) — via {HIGHLIGHT}{agval(author)}{RESET}")


🏛️ Most connected authors in the graph:

    24 books — Agatha Christie
    10 books — C. S. Lewis
     6 books — Janet Evanovich
     5 books — Aldous Huxley
     5 books — Christopher Moore
     5 books — Clive Staples Lewis
     5 books — John Ronald Reuel Tolkien
     5 books — Mary Stewart
     5 books — Meg Cabot
     5 books — Neal Stephenson

🔗 Two hops from 'A Murder is Announced' (book → author → book):

📘 An Autobiography (1977) — via Agatha Christie
📘 Witness for the Prosecution & Selected Plays (1995) — via Agatha Christie
📘 Miss Marple (1997) — via Agatha Christie
📘 Spider's Web (2000) — via Agatha Christie
📘 Appointment with Death (2001) — via Agatha Christie
📘 Death in the Clouds (2001) — via Agatha Christie
📘 Hallowe'en Party (2001) — via Agatha Christie
📘 Hercule Poirot's Christmas (2001) — via Agatha Christie
📘 Murder in Mesopotamia (2001) — via Agatha Christie
📘 Partners in Crime (2001) — via Agatha Christie
📘 The Secret of Chimneys (2001) — via Agatha Christie
📘 En

#### 3.6.2 🎯 Vectors Find, the Graph Expands

The payoff, and the reason both live in one database. A vector search picks the **entry point** —
the book closest in meaning to a free-text phrase — and the graph then **expands outward** from it:
first to semantically similar books, then to everything else those books' authors wrote.

Vector search alone can't reach that second hop. A `JOIN` alone can't find the entry point.
This is the shape most GraphRAG pipelines end up building, in about fifteen lines of query.


In [178]:
# Vector search picks the entry point, then the graph expands outward from it
with postgres_connection.cursor() as cursor:
    # Restricted to the same scoped set, so the winner is guaranteed to exist as a Book node
    cursor.execute(sql.SQL("""
        WITH scoped AS (
            SELECT id, title, embeddings FROM {table}
            WHERE embeddings IS NOT NULL ORDER BY id LIMIT %s
        )
        SELECT id, title FROM scoped
        ORDER BY embeddings <=> azure_openai.create_embeddings(%s, %s)::vector
        LIMIT 1;
    """).format(table=sql.Identifier(POSTGRES_TABLE)),
        (GRAPH_BOOK_LIMIT, FOUNDRY_EMBEDDING_SMALL_MODEL, postgres_search_phrase))
    entry_id, entry_title = cursor.fetchone()

    # OPTIONAL MATCH keeps neighbours whose authors wrote nothing else in this graph
    expansion = run_cypher(cursor, """
        MATCH (seed:Book {id: $id})-[s:SIMILAR_TO]->(similar:Book)
        OPTIONAL MATCH (similar)<-[:WROTE]-(a:Author)-[:WROTE]->(also:Book)
        WHERE also <> seed AND also <> similar
        RETURN DISTINCT similar.title, s.score, a.name, also.title
        ORDER BY s.score DESC
    """, {"id": entry_id}, returns=("similar", "score", "author", "also"))

# Collapse the join fan-out back into one block per similar book
neighbourhood = {}
for similar, score, author, also in expansion:
    entry = neighbourhood.setdefault((agval(similar), agval(score)), [])
    if (title := agval(also)) and (line := f"{title} — {agval(author)}") not in entry:
        entry.append(line)

print(f"🔍 '{postgres_search_phrase}'\n   ↳ closest book: {HIGHLIGHT}{entry_title}{RESET}\n")
for (title, score), also_wrote in neighbourhood.items():
    print(f"📘 {title}  ({HIGHLIGHT}similarity: {score:.4f}{RESET})")
    for line in also_wrote or ["(no other books by this author in the graph)"]:
        print(f"      ↳ {line}")
    print()


🔍 'a lighthearted story about friendship and adventure'
   ↳ closest book: Bridge to Terabithia (rack)

📘 Behind the Curtain  (similarity: 0.4213)
      ↳ (no other books by this author in the graph)

📘 The Kindness of Strangers  (similarity: 0.4172)
      ↳ (no other books by this author in the graph)

📘 The Whipping Boy  (similarity: 0.4142)
      ↳ (no other books by this author in the graph)

📘 Charms for the Easy Life  (similarity: 0.3990)
      ↳ (no other books by this author in the graph)

📘 Where Rainbows End  (similarity: 0.3831)
      ↳ (no other books by this author in the graph)



#### 3.6.3 🖼️ Rendering the Neighbourhood

The same query, drawn instead of printed. The result is laid out as inline SVG — the entry point in
the middle, its semantic neighbours on the inner ring with their similarity scores on the edges, and
the authors those books share on the outer ring.

It is deliberately dependency-free: no JavaScript, no CDN, no extra package. Labels inherit
`currentColor`, so it stays readable in both light and dark notebook themes.


In [179]:
# Draw the same result as inline SVG -- no JavaScript, no CDN, no extra package
import math
from html import escape

from IPython.display import HTML, display

with postgres_connection.cursor() as cursor:
    graph_rows = run_cypher(cursor, """
        MATCH (:Book {id: $id})-[s:SIMILAR_TO]->(similar:Book)
        OPTIONAL MATCH (similar)<-[:WROTE]-(a:Author)
        RETURN DISTINCT similar.title, s.score, a.name
        ORDER BY s.score DESC
    """, {"id": entry_id}, returns=("title", "score", "author"))

# One entry per neighbour; a book with two authors arrives as two rows
ring = {}
for title, score, author in graph_rows:
    entry = ring.setdefault(agval(title), {"score": agval(score), "authors": []})
    if (name := agval(author)) and name not in entry["authors"]:
        entry["authors"].append(name)

W, H, CX, CY = 1000, 700, 500, 350
FILLS = {"Entry point": "#e0af68", "SIMILAR_TO": "#7aa2f7", "Author": "#9ece6a"}


def node(x, y, radius, kind, text, size, anchor="middle", limit=28):
    caption = text if len(text) <= limit else text[:limit - 1] + "…"
    # currentColor keeps labels legible on both light and dark notebook themes
    return (f'<circle cx="{x:.0f}" cy="{y:.0f}" r="{radius}" fill="{FILLS[kind]}" />'
            f'<text x="{x:.0f}" y="{y - radius - 7:.0f}" text-anchor="{anchor}" fill="currentColor" '
            f'font-size="{size}" font-family="system-ui,sans-serif">{escape(caption)}</text>')


def line(x1, y1, x2, y2, width, dashed=False):
    dash = ' stroke-dasharray="4 3"' if dashed else ""
    return (f'<line x1="{x1:.0f}" y1="{y1:.0f}" x2="{x2:.0f}" y2="{y2:.0f}" stroke="#8b93a7" '
            f'stroke-width="{width:.1f}" stroke-opacity="0.5"{dash} />')


def caption(x, y, text, size=11, opacity=0.75):
    return (f'<text x="{x:.0f}" y="{y:.0f}" text-anchor="middle" fill="currentColor" '
            f'font-size="{size}" opacity="{opacity}">{text}</text>')


edges, nodes = [], []

for index, (title, data) in enumerate(ring.items()):
    angle = 2 * math.pi * index / len(ring) - math.pi / 2
    bx, by = CX + 180 * math.cos(angle), CY + 180 * math.sin(angle)

    # Thicker edge for a closer book, so the ranking reads before the numbers do
    edges.append(line(CX, CY, bx, by, 1 + 5 * max(data["score"], 0)))
    edges.append(caption(CX + (bx - CX) * .55, CY + (by - CY) * .55, f'{data["score"]:.3f}'))
    nodes.append(node(bx, by, 11, "SIMILAR_TO", title, 12))

    for offset, author in enumerate(data["authors"]):
        # Fan multiple authors apart so their labels don't collide
        spread = angle + (offset - (len(data["authors"]) - 1) / 2) * 0.17
        ax, ay = CX + 300 * math.cos(spread), CY + 300 * math.sin(spread)
        edges.append(line(bx, by, ax, ay, 1, dashed=True))
        nodes.append(node(ax, ay, 7, "Author", author, 11, "start" if ax >= CX else "end", 24))

nodes.append(node(CX, CY, 16, "Entry point", entry_title, 13))

swatch = "display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:6px"
legend = " ".join(f'<span><span style="{swatch};background:{fill}"></span>{kind}</span>'
                  for kind, fill in FILLS.items())

display(HTML(f'<div style="font-family:system-ui,sans-serif;font-size:13px">'
             f'<div style="display:flex;gap:1.5rem;margin:0 0 .25rem .5rem">{legend}</div>'
             f'<svg viewBox="0 0 {W} {H}" width="100%">{"".join(edges)}{"".join(nodes)}</svg></div>'))


#### 3.6.4 🧩 The Same Graph in the PostgreSQL Extension

The SVG above is hand-drawn because the notebook queries return **scalar properties** (`b.title`,
`s.score`). The [PostgreSQL extension for VS Code](https://learn.microsoft.com/azure/postgresql/development/vs-code-extension/postgresql-extension-overview)
renders an AGE result as an interactive node-edge graph for free, provided the query meets
[three documented requirements](https://learn.microsoft.com/azure/postgresql/azure-ai/generative-ai-age-overview):

1. **Return whole vertices and edges** (`RETURN a, r, b`) — a query returning `b.title` produces
   plain text and reports *"The query returned no graphable data"*.
2. **Set `disp_label`** on each node, or the visualizer shows internal IDs instead of names.
   The build cell above already does this.
3. **One `AS (...)` column per returned object**, including every intermediate node and edge.

Open a `.sql` editor connected to this same database and run the following. Literals are fine
*inside* the Cypher body — only `cypher()`'s third argument must be a bind parameter, so none of
these need `PREPARE`.

```sql
-- Per connection: ag_catalog holds cypher() and the agtype operators
SET search_path = ag_catalog, "$user", public;
```

**Sanity check first** — scalar output, so it will *not* render, but it proves the graph has data.
If either count is `0`, re-run the two build cells above:

```sql
SELECT * FROM cypher('books_graph', $$
    MATCH (n) RETURN count(n)
$$) AS ("nodes" agtype);

SELECT * FROM cypher('books_graph', $$
    MATCH ()-[r:SIMILAR_TO]->() RETURN count(r)
$$) AS ("similar_to_edges" agtype);
```

**1. The semantic layer** — self-seeding, so it renders without editing anything:

```sql
SELECT * FROM cypher('books_graph', $$
    MATCH (b:Book)-[s:SIMILAR_TO]->(n:Book)
    RETURN b, s, n
    LIMIT 40
$$) AS ("b" agtype, "s" agtype, "n" agtype);
```

**2. Authors joined in** — plain `MATCH` rather than `OPTIONAL MATCH`, so no column comes back null:

```sql
SELECT * FROM cypher('books_graph', $$
    MATCH (a:Author)-[w:WROTE]->(b:Book)-[s:SIMILAR_TO]->(n:Book)
    RETURN a, w, b, s, n
    LIMIT 25
$$) AS ("a" agtype, "w" agtype, "b" agtype, "s" agtype, "n" agtype);
```

**3. One book's neighbourhood** — the shape the SVG draws. Only the first 300 books are in the
graph, so take a title that is actually present rather than guessing one:

```sql
-- Scalar result, will not render -- it is just here to give you a valid title
SELECT * FROM cypher('books_graph', $$
    MATCH (b:Book)-[:SIMILAR_TO]->() RETURN b.title LIMIT 10
$$) AS ("title" agtype);
```

```sql
SELECT * FROM cypher('books_graph', $$
    MATCH (seed:Book {title: 'PASTE A TITLE FROM ABOVE'})-[s:SIMILAR_TO]->(n:Book)
    MATCH (n)<-[w:WROTE]-(a:Author)
    RETURN seed, s, n, w, a
$$) AS ("seed" agtype, "s" agtype, "n" agtype, "w" agtype, "a" agtype);
```

> ⚠️ Output column names are quoted for a reason — `similar`, `count`, `order` and `type` are all
> PostgreSQL reserved words and fail with a bare `syntax error` if left unquoted.

> 🏷️ Built the graph before `disp_label` was added? Backfill it without rebuilding:
> ```sql
> SELECT * FROM cypher('books_graph', $$ MATCH (b:Book)   SET b.disp_label = b.title $$) AS ("r" agtype);
> SELECT * FROM cypher('books_graph', $$ MATCH (n:Author) SET n.disp_label = n.name  $$) AS ("r" agtype);
> ```


#### 3.6.5 🖼️ Rendered by the Extension

`books_graph_similar` — the semantic layer on its own: every edge here was computed from the embeddings, not from a foreign key.

<img src="../assets/graph-books-similarity.png" alt="SIMILAR_TO graph rendered in the PostgreSQL extension" width="100%">

`books_graph_authors` — the same graph with authors joined in, so the semantic clusters resolve into who actually wrote what.

<img src="../assets/graph-books-authros.png" alt="Author and SIMILAR_TO graph rendered in the PostgreSQL extension" width="100%">


### 3.7 📚 References

**Extensions**

- [Allow extensions](https://learn.microsoft.com/azure/postgresql/extensions/how-to-allow-extensions) — the `azure.extensions` server allow-list, separate from `CREATE EXTENSION`.
- [Enable and use pgvector](https://learn.microsoft.com/azure/postgresql/extensions/how-to-use-pgvector) — the `vector` type and the `<->`, `<#>`, `<=>` distance operators.
- [Enable and use DiskANN](https://learn.microsoft.com/azure/postgresql/extensions/how-to-use-pgdiskann) — `pg_diskann` index creation, operator classes, and tuning parameters.

**In-database AI**

- [azure_ai extension overview](https://learn.microsoft.com/azure/postgresql/azure-ai/generative-ai-azure-overview) — schemas, `set_setting` / `get_setting`, and configuration keys.
- [Generate vector embeddings with Azure OpenAI](https://learn.microsoft.com/azure/postgresql/azure-ai/generative-ai-azure-openai) — full `azure_openai.create_embeddings` signature, including `dimensions`, `batch_size`, `max_attempts`, and `retry_delay_ms`.
- [Tutorial: semantic search with Azure Database for PostgreSQL and Azure OpenAI](https://learn.microsoft.com/azure/postgresql/azure-ai/generative-ai-semantic-search) — end-to-end walkthrough of the pattern this section demonstrates.

**Graph**

- [AGE extension with Azure Database for PostgreSQL](https://learn.microsoft.com/azure/postgresql/azure-ai/generative-ai-age-overview) — enabling `age`, the `ag_catalog` schema, `ag_graph` / `ag_label`, and the `AS (...)` output rules.
- [Extension considerations](https://learn.microsoft.com/azure/postgresql/extensions/concepts-extensions-considerations) — why `age` needs `shared_preload_libraries` as well as the allow-list.
- [Load libraries](https://learn.microsoft.com/azure/postgresql/extensions/how-to-load-libraries) — setting `shared_preload_libraries`, which restarts the server.
- [Extension versions](https://learn.microsoft.com/azure/postgresql/extensions/concepts-extensions-versions) — which AGE version ships with each PostgreSQL major version.
- [PostgreSQL extension for Visual Studio Code](https://learn.microsoft.com/azure/postgresql/development/vs-code-extension/postgresql-extension-overview) — renders Cypher results as an interactive node-edge graph.


---

## 4. 🧹 Cleanup


The Foundry embedding clients are **async** and hold an `aiohttp` session each. Left open, Python
eventually reports `Unclosed client session` — usually long after the cell that created them, which
makes it look unrelated.

Run this last, or any time before shutting the kernel down.

In [180]:
# Release the async HTTP sessions and the database connections
if "embedding_client" in globals():
    await embedding_client.close()
if "text_client" in globals():
    await text_client.close()

# FoundryChatClient is absent on purpose: it exposes no close(), holding no session of its own
if "cosmos_client" in globals():
    cosmos_client.close()

if "postgres_connection" in globals() and not postgres_connection.closed:
    postgres_connection.close()

print("✅ Clients and connections closed.")


✅ Clients and connections closed.
